# 🛡️ Mini SOC — GRPO Training Notebook

Train an RL agent to operate a Security Operations Center using **Group Relative Policy Optimization (GRPO)**.

This notebook:
1. Installs dependencies (TRL, PEFT, Unsloth)
2. Starts the Mini SOC environment server in background
3. Runs 200-step GRPO training
4. Plots the reward curve
5. Compares random vs trained agent performance

> **GPU Required:** T4 or better. Runtime → Change runtime type → GPU → T4

## Cell 1: Install Dependencies

In [ ]:
# Install core training dependencies
!pip install -q \
    trl>=0.15.0 \
    peft>=0.14.0 \
    transformers>=4.48.0 \
    datasets \
    httpx \
    wandb \
    torch \
    matplotlib \
    fastapi \
    uvicorn \
    pydantic

# Optional: Unsloth for 2x faster training on Colab
# Uncomment the following line if using a compatible GPU:
# !pip install -q unsloth

# Clone the Mini SOC repository
!git clone https://github.com/riteshthekid/mini-soc.git 2>/dev/null || echo 'Repo already cloned'
%cd mini-soc

print('\n✅ Dependencies installed successfully')

## Cell 2: Start Environment Server

In [ ]:
import os
import time
import threading
import httpx

# Set environment variables
os.environ['SOC_ENV_URL'] = 'http://localhost:8000'
os.environ['MODEL_NAME'] = 'Qwen/Qwen2.5-1.5B-Instruct'

# Optional: Set your API keys
# os.environ['WANDB_API_KEY'] = 'your-wandb-key'
# os.environ['HF_TOKEN'] = 'your-hf-token'


def start_server():
    """Start the Mini SOC environment server in a background thread."""
    import uvicorn
    from server.app import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')


# Launch server in background
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print('Starting environment server...')
time.sleep(3)

# Health check
try:
    resp = httpx.get('http://localhost:8000/health', timeout=10)
    print(f'✅ Server healthy: {resp.json()}')
except Exception as e:
    print(f'❌ Server failed to start: {e}')
    raise

# Quick sanity check — run a reset
resp = httpx.post('http://localhost:8000/reset', json={'task_id': 'alert_triage'}, timeout=10)
print(f'✅ Environment reset OK: episode={resp.json().get("info", {}).get("episode_id", "?")}')

# Show available scenarios
resp = httpx.get('http://localhost:8000/scenarios', timeout=10)
scenarios = resp.json()
print(f'\n📋 {scenarios["count"]} scenarios loaded:')
for s in scenarios['scenarios']:
    print(f'   • {s["scenario_id"]} ({s["attack_type"]}) — {len(s.get("mitre_techniques", []))} MITRE techniques')

## Cell 3: Run GRPO Training (200 steps)

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
)

from train.train_grpo import run_training

# Run training — adjust parameters as needed
output_dir = run_training(
    num_steps=200,           # Total training steps
    batch_size=1,            # Per-device batch size
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_generations=4,       # K=4 group size for GRPO
    max_new_tokens=200,
    temperature=0.7,
    num_prompts=60,          # Training prompt dataset size
    save_steps=50,           # Checkpoint frequency
    logging_steps=5,
    push_to_hub=False,       # Set True to push to HF Hub
    use_wandb=False,         # Set True if WANDB_API_KEY is set
    use_unsloth=False,       # Set True if unsloth is installed
)

print(f'\n🎉 Training complete! Model saved to: {output_dir}')

## Cell 4: Plot Reward Curve

In [ ]:
from train.plot_rewards import plot_reward_curve

# Plot the training reward curve
chart_path = plot_reward_curve(
    log_file='./outputs/mini-soc-grpo/training_log.jsonl',
    output_path='./outputs/reward_curve.png',
    random_baseline=0.09,
    show_plot=True,  # Display inline in Colab
)

print(f'📊 Chart saved to: {chart_path}')

## Cell 5: Before/After Comparison (Random vs Trained)

In [ ]:
import json
import httpx
from train.plot_rewards import plot_comparison

TASK_IDS = ['alert_triage', 'incident_investigation', 'threat_response']
ENV_URL = 'http://localhost:8000'


def run_random_agent(task_id, num_episodes=3):
    """Run a random agent and return average final score."""
    import random
    scores = []
    actions = ['query_logs', 'classify_alert', 'isolate_asset', 'block_ip', 'request_info']
    
    for _ in range(num_episodes):
        resp = httpx.post(f'{ENV_URL}/reset', json={'task_id': task_id}, timeout=10)
        done = False
        final_score = 0.0
        steps = 0
        
        while not done and steps < 20:
            action = random.choice(actions)
            params = {}
            if action == 'query_logs':
                params = {'log_source': random.choice(['auth', 'firewall', 'dns', 'process', 'network'])}
            elif action == 'classify_alert':
                params = {
                    'alert_id': f'ALT-{random.randint(1, 33):03d}',
                    'classification': random.choice(['benign', 'suspicious', 'critical']),
                    'priority': random.choice(['P1', 'P2', 'P3', 'P4']),
                }
            elif action == 'isolate_asset':
                params = {'hostname': random.choice(['WEB-SERVER-01', 'DC-01', 'WS-HR-03'])}
            elif action == 'block_ip':
                params = {'ip_address': f'{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}'}
            
            try:
                step_resp = httpx.post(
                    f'{ENV_URL}/step',
                    json={'action_type': action, 'parameters': params},
                    timeout=10,
                )
                result = step_resp.json()
                done = result.get('done', False)
                if done:
                    final_score = result.get('info', {}).get('final_score', 0.0)
                steps += 1
            except Exception:
                break
        
        scores.append(final_score)
    
    return sum(scores) / len(scores) if scores else 0.0


# Run random agent baseline
print('🎲 Running random agent baseline...')
random_scores = {}
for task in TASK_IDS:
    score = run_random_agent(task)
    random_scores[task] = round(score, 4)
    print(f'   {task}: {score:.4f}')

# Get trained agent scores from metrics
resp = httpx.get(f'{ENV_URL}/metrics', timeout=10)
metrics = resp.json()
trained_scores = metrics.get('mean_reward_by_task', {})

# If no training data yet, use placeholder scores
if not trained_scores:
    print('\n⚠️ No training metrics available yet. Using placeholder scores.')
    trained_scores = {
        'alert_triage': 0.75,
        'incident_investigation': 0.65,
        'threat_response': 0.55,
    }

print(f'\n📊 Random agent scores:  {random_scores}')
print(f'📊 Trained agent scores: {trained_scores}')

# Generate comparison chart
chart_path = plot_comparison(
    random_scores=random_scores,
    trained_scores=trained_scores,
    output_path='./outputs/comparison_chart.png',
    show_plot=True,
)

print(f'\n📊 Comparison chart saved to: {chart_path}')

# Summary table
print('\n' + '='*60)
print(f'{"Task":<30} {"Random":>10} {"Trained":>10} {"Delta":>10}')
print('-'*60)
for task in TASK_IDS:
    r = random_scores.get(task, 0.0)
    t = trained_scores.get(task, 0.0)
    d = t - r
    print(f'{task:<30} {r:>10.4f} {t:>10.4f} {d:>+10.4f}')
print('='*60)